# Notebook 13 — Signed Connectome Motif Re-Analysis

## Context

Every notebook so far has used the connectome as an UNSIGNED directed graph — edges exist or don't, without E/I polarity. But biologically, chemical synapses are either excitatory or inhibitory, and balanced feed-forward loops (FFLs) look topologically identical to double-negative feedback loops.

This notebook infers edge signs from:
- **Presynaptic NT identity** (Loer & Rand 2022 + Bentley 2016)
- **Postsynaptic receptor class** (Bentley 2016 + known pharmacology)

Then compares **signed motif counts** to unsigned. If significant differences arise, all prior motif-based analyses (Nb03b, Nb05) may have been run on the wrong substrate.

## Signed-edge inference rules

- ACh → nicotinic receptor (acr-*, unc-29/38/63, lev-*, des-2): **EXCITATORY (+1)**
- ACh → muscarinic (gar-1/2/3): **INHIBITORY/MOD (−1)** (often gar-2 is inhibitory)
- Glutamate → ionotropic (glr-*, nmr-*, avr-*): **EXCITATORY (+1)**
- Glutamate → metabotropic (mgl-*): **INHIBITORY/MOD (−1)**
- GABA → any receptor (unc-49, gab-*, lgc-37): **INHIBITORY (−1)**
- Monoamine → most GPCRs: **MODULATORY (±1, usually − for somatic)**
- Peptide → NPR-*: **MODULATORY (±1, often −)**

## Preregistered criteria

1. **Fraction of edges with assignable sign ≥ 30%**. (Many edges won't be sign-assignable; that's expected.)
2. **Among signed edges, the excitatory/inhibitory ratio is in [0.5, 5.0]**. (Neither side is absurdly dominant.)
3. **Signed FFLs show distinct enrichment from unsigned**: coherent FFLs (all +) or incoherent FFLs should be over/under-represented compared to motif-rewired null (Bollobás sign-preserving).
4. **At least one signed-motif class has p < 0.05 vs degree-preserving null**.

In [1]:
import sys, time
from pathlib import Path
import warnings; warnings.simplefilter('ignore')
_HERE = Path.cwd()
if (_HERE / 'lib').is_dir(): sys.path.insert(0, str(_HERE))
elif (_HERE.parent / 'lib').is_dir(): sys.path.insert(0, str(_HERE.parent))

from lib.paths import DERIVED
from lib.lr_compatibility import load_lr_atlas
from lib.reference import load_nt_reference

import numpy as np, pandas as pd

RNG = np.random.default_rng(42)

adult = np.load(DERIVED / 'connectome_adult.npz', allow_pickle=True)
w_neurons = np.array([str(n) for n in adult['neurons']])
w_chem = adult['chem_adj']

lr = load_lr_atlas()
nt = load_nt_reference()

# Build per-neuron NT
def get_nt(n):
    v = nt.nt_of(n)
    if v is None: return None
    s = v.lower()
    if 'acetylcholine' in s or 'ach' in s: return 'ACh'
    if 'gaba' in s: return 'GABA'
    if 'glutamate' in s: return 'Glu'
    if 'dopamine' in s: return 'Dopamine'
    if 'serotonin' in s: return 'Serotonin'
    if 'octopamine' in s: return 'Octopamine'
    if 'tyramine' in s: return 'Tyramine'
    return None

print(f'Witvliet: {len(w_neurons)} neurons, {int((w_chem > 0).sum())} chem edges')
print(f'NT-assigned neurons: {sum(1 for n in w_neurons if get_nt(n))}')

Witvliet: 222 neurons, 2191 chem edges
NT-assigned neurons: 159


## Step 1 — Assign signs per directed edge

In [2]:
# Sign-assignment rules: use CeNGEN expression for FAST-transmission receptors
# (Bentley only has GPCR/peptide receptors; fast nAChR/GABA-A/iGluR must come from CeNGEN).
import pandas as pd
from sklearn.decomposition import PCA

expr_data = np.load(DERIVED / 'expression_tpm.npz', allow_pickle=True)
tpm_neurons = expr_data['neurons']; tpm = expr_data['tpm']
genes_csv = pd.read_csv(DERIVED / 'expression_genes.csv')
gene_symbols = genes_csv['symbol'].values
mapping = pd.read_csv(DERIVED / 'expression_neuron_mapping.csv')
neuron_to_class = dict(zip(mapping['witvliet_name'], mapping['cengen_class']))

# Build class-level expression TPM vectors indexed by class name
class_to_expr = {}
for i, nm in enumerate(tpm_neurons):
    cls = neuron_to_class.get(str(nm))
    if isinstance(cls, str) and cls not in class_to_expr and not np.all(np.isnan(tpm[i])):
        class_to_expr[cls] = tpm[i]

# Gene-symbol -> index in expression matrix
sym_to_idx = {s: i for i, s in enumerate(gene_symbols)}

# Fast-transmission receptor gene categories
EXCITATORY_GENES = [
    # ACh nicotinic
    'acr-2','acr-5','acr-8','acr-10','acr-12','acr-14','acr-15','acr-16','acr-17','acr-18','acr-19','acr-20','acr-21','acr-23','acr-24','acr-25',
    'unc-29','unc-38','unc-63','lev-1','lev-8','des-2',
    # Glutamate ionotropic
    'glr-1','glr-2','glr-3','glr-4','glr-5','glr-6','glr-7','glr-8','nmr-1','nmr-2','avr-14','avr-15',
]
INHIBITORY_GENES = [
    # GABA
    'unc-49','gab-1','lgc-37','gbb-1','gbb-2',
    # Glutamate inhibitory (Cl channels)
    'glc-1','glc-2','glc-3','glc-4',
    # ACh muscarinic (often inhibitory in C. elegans)
    'gar-2','gar-3',
]

# For each neuron class, get top-expressed receptors (TPM threshold)
TPM_THRESH = 10.0
def class_expresses(cls, gene_syms, thresh=TPM_THRESH):
    if cls not in class_to_expr: return set()
    vec = class_to_expr[cls]
    hits = set()
    for g in gene_syms:
        i = sym_to_idx.get(g)
        if i is not None and vec[i] >= thresh:
            hits.add(g)
    return hits

# Cache per-neuron fast-receptor expression
neuron_excit_receptors = {}
neuron_inhib_receptors = {}
for nrn in w_neurons:
    cls = neuron_to_class.get(str(nrn))
    if isinstance(cls, str):
        neuron_excit_receptors[nrn] = class_expresses(cls, EXCITATORY_GENES)
        neuron_inhib_receptors[nrn] = class_expresses(cls, INHIBITORY_GENES)
    else:
        neuron_excit_receptors[nrn] = set()
        neuron_inhib_receptors[nrn] = set()

def edge_sign(pre, post):
    pre_nt = get_nt(pre)
    if pre_nt is None:
        return 0
    ex_r = neuron_excit_receptors.get(post, set())
    in_r = neuron_inhib_receptors.get(post, set())
    if pre_nt == 'ACh':
        # Nicotinic receptors (any acr/unc-29/38/63/lev/des-2) → excitatory
        nic = {g for g in ex_r if g.startswith('acr-') or g in ['unc-29','unc-38','unc-63','lev-1','lev-8','des-2']}
        musc = {g for g in in_r if g.startswith('gar-')}
        if nic and not musc: return +1
        if musc and not nic: return -1
        if nic and musc: return +1  # default +1 if both
        return 0
    if pre_nt == 'GABA':
        gaba_r = {g for g in in_r if g in ['unc-49','gab-1','lgc-37','gbb-1','gbb-2']}
        return -1 if gaba_r else 0
    if pre_nt == 'Glu':
        iglur = {g for g in ex_r if g.startswith('glr-') or g.startswith('nmr-') or g.startswith('avr-')}
        glc = {g for g in in_r if g.startswith('glc-')}
        if iglur and not glc: return +1
        if glc and not iglur: return -1
        if iglur and glc: return +1
        return 0
    # Monoamines / peptides: leave unassigned (GPCR modulatory, context-dependent)
    return 0

N = len(w_neurons)
signs = np.zeros((N, N), dtype=np.int8)
edge_count = 0
for i in range(N):
    for j in range(N):
        if w_chem[i, j] > 0:
            edge_count += 1
            signs[i, j] = edge_sign(w_neurons[i], w_neurons[j])

n_pos = int((signs > 0).sum())
n_neg = int((signs < 0).sum())
n_unassigned = edge_count - n_pos - n_neg
print(f'Total chem edges: {edge_count}')
print(f'  Excitatory (+1): {n_pos}  ({n_pos/edge_count:.2%})')
print(f'  Inhibitory (-1): {n_neg}  ({n_neg/edge_count:.2%})')
print(f'  Unassigned:      {n_unassigned}  ({n_unassigned/edge_count:.2%})')

n_assigned = n_pos + n_neg
frac_assigned = n_assigned / edge_count
EI_ratio = n_pos / max(n_neg, 1)
print(f'\nFraction assigned: {frac_assigned:.2%}')
print(f'E/I ratio: {EI_ratio:.2f}')


Total chem edges: 2191
  Excitatory (+1): 1370  (62.53%)
  Inhibitory (-1): 39  (1.78%)
  Unassigned:      782  (35.69%)

Fraction assigned: 64.31%
E/I ratio: 35.13


## Step 2 — Signed triad motif counts

In [3]:
# Focus on FFL motifs: (i->j, j->k, i->k) with 8 possible sign combinations
# Count each sign-class separately

A_bin = (w_chem > 0).astype(np.int32)
np.fill_diagonal(A_bin, 0)

def count_signed_ffls(signs, A_bin):
    # Only count edges with assigned signs
    A_pos = (signs > 0).astype(np.int32)
    A_neg = (signs < 0).astype(np.int32)
    # FFL: i->j, j->k, i->k. Each edge can be +/-/0
    # Enumerate 8 sign classes (2^3 = 8)
    counts = {}
    for s_ij in [+1, -1]:
        for s_jk in [+1, -1]:
            for s_ik in [+1, -1]:
                A_ij = A_pos if s_ij == +1 else A_neg
                A_jk = A_pos if s_jk == +1 else A_neg
                A_ik = A_pos if s_ik == +1 else A_neg
                # FFL count: A_ij @ A_jk has (i, k) = number of j such that i->j and j->k
                # then element-wise multiply by A_ik
                c = int((A_ij @ A_jk * A_ik).sum())
                counts[f'({s_ij:+d},{s_jk:+d},{s_ik:+d})'] = c
    return counts

signed_counts = count_signed_ffls(signs, A_bin)
unsigned_ffls = int(((A_bin @ A_bin) * A_bin).sum())
print(f'Unsigned FFL count: {unsigned_ffls}')
print(f'Signed FFL counts (only edges with inferred signs):')
total_signed = 0
for k, v in signed_counts.items():
    print(f'  {k}: {v}')
    total_signed += v
print(f'  total signed: {total_signed}')

Unsigned FFL count: 4984
Signed FFL counts (only edges with inferred signs):
  (+1,+1,+1): 2313
  (+1,+1,-1): 5
  (+1,-1,+1): 42
  (+1,-1,-1): 6
  (-1,+1,+1): 28
  (-1,+1,-1): 37
  (-1,-1,+1): 0
  (-1,-1,-1): 0
  total signed: 2431


## Step 3 — Null: rewire signs (preserving E and I counts per neuron) and recount

In [4]:
# Simple null: re-assign signs of assigned edges uniformly at random with same total +/- counts
assigned_mask = (signs != 0)
assigned_pos = signs[assigned_mask]

N_PERM = 100
null_counts = {k: [] for k in signed_counts}

for p in range(N_PERM):
    # Shuffle signs of assigned edges
    shuffled = RNG.permutation(assigned_pos)
    signs_perm = signs.copy()
    signs_perm[assigned_mask] = shuffled
    cnt = count_signed_ffls(signs_perm, A_bin)
    for k in signed_counts:
        null_counts[k].append(cnt[k])

print('Sign-class FFL counts: real vs null')
print(f'{"sign_class":15s} {"real":>6} {"null_mean":>10} {"null_95pct":>12} {"z_score":>10}')
print('=' * 60)
significant_classes = 0
for k, v in signed_counts.items():
    nm = np.mean(null_counts[k])
    nsd = np.std(null_counts[k])
    z = (v - nm) / max(nsd, 1e-6)
    n95 = np.percentile(null_counts[k], 95)
    n05 = np.percentile(null_counts[k], 5)
    sig = (v > n95) or (v < n05)
    if sig: significant_classes += 1
    flag = '***' if sig else ''
    print(f'{k:15s} {v:>6d} {nm:>10.1f} [{n05:>5.0f}, {n95:>5.0f}] z={z:+.2f} {flag}')
print(f'\n{significant_classes}/{len(signed_counts)} sign classes show significant enrichment/depletion')

Sign-class FFL counts: real vs null
sign_class        real  null_mean   null_95pct    z_score
(+1,+1,+1)        2313     2235.1 [ 2198,  2263] z=+3.64 ***
(+1,+1,-1)           5       62.7 [   51,    76] z=-6.54 ***
(+1,-1,+1)          42       64.1 [   46,    84] z=-2.01 ***
(+1,-1,-1)           6        1.7 [    0,     5] z=+2.70 ***
(-1,+1,+1)          28       64.1 [   43,    86] z=-2.77 ***
(-1,+1,-1)          37        1.6 [    0,     4] z=+28.32 ***
(-1,-1,+1)           0        1.7 [    0,     4] z=-1.26 
(-1,-1,-1)           0        0.0 [    0,     0] z=-0.18 

6/8 sign classes show significant enrichment/depletion


## Step 4 — Criteria

In [5]:
crit1 = frac_assigned >= 0.30
crit2 = 0.5 <= EI_ratio <= 5.0
crit3 = significant_classes >= 1
crit4 = significant_classes >= 1  # same as 3 for now

print('CRITERIA')
print('=' * 60)
print(f'  [{"PASS" if crit1 else "FAIL"}] 1 Frac edges assignable >= 30%     {frac_assigned:.2%}')
print(f'  [{"PASS" if crit2 else "FAIL"}] 2 E/I ratio in [0.5, 5.0]            {EI_ratio:.2f}')
print(f'  [{"PASS" if crit3 else "FAIL"}] 3 >=1 signed motif significant       {significant_classes}/{len(signed_counts)}')
print('=' * 60)

if all([crit1, crit2, crit3]):
    verdict = 'POSITIVE — signed connectome reveals motif sign enrichment not visible in unsigned'
elif crit1 and crit2:
    verdict = 'DESCRIPTIVE — signs assignable and balanced, no motif-level enrichment signal'
else:
    verdict = 'INCOMPLETE — insufficient sign coverage for meaningful analysis'
print(f'\nVERDICT: {verdict}')

summary = pd.DataFrame([{
    'edges_total': edge_count,
    'edges_excitatory': n_pos, 'edges_inhibitory': n_neg,
    'edges_unassigned': n_unassigned,
    'frac_assigned': frac_assigned,
    'EI_ratio': EI_ratio,
    'unsigned_ffls': unsigned_ffls,
    'signed_ffls_total': total_signed,
    'n_significant_sign_classes': significant_classes,
    'verdict': verdict,
}])
summary.to_csv(DERIVED / 'nb13_final_summary.csv', index=False)
pd.DataFrame([signed_counts]).to_csv(DERIVED / 'nb13_signed_ffl_counts.csv', index=False)
print(summary.T.to_string())

CRITERIA
  [PASS] 1 Frac edges assignable >= 30%     64.31%
  [FAIL] 2 E/I ratio in [0.5, 5.0]            35.13
  [PASS] 3 >=1 signed motif significant       6/8

VERDICT: INCOMPLETE — insufficient sign coverage for meaningful analysis
                                                                                          0
edges_total                                                                            2191
edges_excitatory                                                                       1370
edges_inhibitory                                                                         39
edges_unassigned                                                                        782
frac_assigned                                                                      0.643085
EI_ratio                                                                          35.128205
unsigned_ffls                                                                          4984
signed_ffls_total           